# Temporary Set of Tutorials for Demonstrating Crosslinks

TODO: Convert this into style of other tutorials
* 1. x Basic Crosslink single bead and check positions
* 2. x Crosslink specific sites
* 3. x Crosslinkg all A-C sites
* 4. x Use max backbone connectivity
* 5. x Show different seeds
* 6. x Crosslink full molecule
* 7. x Crosslink around a backbone using bond lengths
* 8. Crosslink then replace backbone beads
* 9. x Demonstrate using excluded bond depth to bond to neighboring sites
* 10. x Crosslink to two neighboring backbone sites
* 11. Internal crosslinks
* 12. Crosslink at high densities and relax structures

In [1]:
from mbuild.path.crosslink import crosslink, CrosslinkerGeometry
from mbuild.exceptions import PathConvergenceError
from mbuild.path.build import Path
from mbuild.path.crosslink import crosslink, CrosslinkerGeometry
import numpy as np
import mbuild as mb
import networkx as nx

In [ ]:
# 1. Crosslink single bead

coordinates = np.array([[0,0,0],[1,1,0], [1,0,0], [0,1,0]])
path = Path(coordinates, bead_name="_B")
path.bond_graph.add_edges_from([[0,2], [1,3]])
cl = CrosslinkerGeometry.single_site("_CROSS")

crosslink(
    path, crosslinker=cl, backbone_name="_B", 
    crosslink_bond_length=np.sqrt(2)/2, 
    tolerance=0.01, seed=1
    )
view = path.visualize(radius=0.5)
view.show()
view.png()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [32]:
view.model

py3Dmol.view.model

In [3]:
# 2. Crosslink specific sites
def build_linear_path(n_sites, y_pos, beads):
    coords = np.array([np.arange(n_sites), np.ones(n_sites)*y_pos, np.zeros(n_sites)]).T
    path = Path(coords, bead_name=beads)
    path._connect_edges(connectivity="linear")
    return path

def build_parallel_linear_paths(n_sites, spacing=[1]):
    beads1 = [chr(ord("A")+i%2) for i in range(n_sites)]
    pathContainer = build_linear_path(n_sites, 0, beads1)

    for offset,space in enumerate(spacing):

        beads2 = [chr(ord("C")+offset*2+i%2) for i in range(n_sites)]
        path2 = build_linear_path(n_sites, space, beads2)
        pathContainer += path2

    return pathContainer

path = build_parallel_linear_paths(5)
path.visualize(radius=0.5)

cl = CrosslinkerGeometry.single_site("_CROSS")
crosslink(
    path, crosslinker=cl, backbone_name=("A", "D"), 
    crosslink_bond_length=np.sqrt(2)/2,
    )
path.visualize(radius=0.4)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [4]:
# 3. Crosslink all possible sites
path = build_parallel_linear_paths(5)

cl = CrosslinkerGeometry.single_site("_CROSS")

for _ in range(10):
    try:
        crosslink(
            path, crosslinker=cl, backbone_name=("A", "D"), 
            crosslink_bond_length=np.sqrt(2)/2,
            )
    except PathConvergenceError:
        break
path.visualize(radius=0.4)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [5]:
# 4. Limit crosslinks by max_backbone degree
path = build_parallel_linear_paths(5)

cl = CrosslinkerGeometry.single_site("_CROSS")
for _ in range(10):
    try:
        crosslink(
            path, crosslinker=cl, backbone_name=("A", "D"), 
            crosslink_bond_length=np.sqrt(2)/2, 
            max_backbone_degree=3, seed=1
            )
    except PathConvergenceError:
        break
path.visualize(radius=0.4)



3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [6]:
# 5. Different random seeds
path = build_parallel_linear_paths(5)

cl = CrosslinkerGeometry.single_site("_CROSS")

for _ in range(10):
    try:
        crosslink(
            path, crosslinker=cl, backbone_name=("A", "D"), 
            crosslink_bond_length=np.sqrt(2)/2, 
            max_backbone_degree=3, seed=2
            )
    except PathConvergenceError:
        break
path.visualize(radius=0.4)



3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
# 6. Crosslink full molecule and find optimal orientation

def strip_hydrogens(cpd):
    """Remove compound hydrogens"""
    atoms_to_remove = []
    for site in cpd.particles():
        if site.name == "H":
            atoms_to_remove.append(site)
    if atoms_to_remove:
        cpd.remove(atoms_to_remove)
    return


path = build_parallel_linear_paths(5, spacing=[2])

mol = mb.load("C=Cc1ccc(C=C)cc1", smiles=True) # divinylbenzene
strip_hydrogens(mol)
dvbPath = Path.from_compound(mol)
cl_distance = np.linalg.norm(dvbPath.coordinates[0] - dvbPath.coordinates[7])
cl = CrosslinkerGeometry.from_path(dvbPath, connection_sites=[0, 7])

for _ in range(10):
    try:
        crosslink(
            path, crosslinker=cl, backbone_name=("B", "D"), 
            crosslink_bond_length=(2-cl_distance)/2, 
            max_backbone_degree=3, tolerance=0.1,
            )
    except PathConvergenceError as e:
        break
path.visualize(0.25)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [8]:
# 7. Crosslink around a chain
import numpy as np
import mbuild as mb
from mbuild.path.build import Path
from mbuild.path.crosslink import crosslink, CrosslinkerGeometry
from mbuild.exceptions import PathConvergenceError


path = build_parallel_linear_paths(5, spacing=[1,2])

mol = mb.load("C=Cc1ccc(C=C)cc1", smiles=True) # divinylbenzene
strip_hydrogens(mol)
dvbPath = Path.from_compound(mol)
cl_distance = np.linalg.norm(dvbPath.coordinates[0] - dvbPath.coordinates[7])
cl = CrosslinkerGeometry.from_path(dvbPath, connection_sites=[0, 7])

for _ in range(10):
    try:
        crosslink(
            path, crosslinker=cl, backbone_name=("B", "F"), 
            crosslink_bond_length=(2-cl_distance)/2*np.sqrt(2),  
            max_backbone_degree=3, tolerance=1,
            )
    except PathConvergenceError as e:
        break
path.visualize(0.25)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
# 8. Crosslink and replace backbone


# TODO: Failing

path = build_parallel_linear_paths(5, spacing=[2])

mol = mb.load("C=Cc1ccc(C=C)cc1", smiles=True) # divinylbenzene
strip_hydrogens(mol)
dvbPath = Path.from_compound(mol)
cl_distance = np.linalg.norm(dvbPath.coordinates[0] - dvbPath.coordinates[7])
cl = CrosslinkerGeometry.from_path(dvbPath, connection_sites=[6, 7],)

for _ in range(10):
    try:
        crosslink(
            path, crosslinker=cl, backbone_name=("B", "D"), 
            crosslink_bond_length=(2-cl_distance)/2, 
            max_backbone_degree=3, tolerance=0.1,
            )
    except PathConvergenceError as e:
        break

PS = mb.load("c1(ccccc1)CC", smiles=True)

strip_hydrogens(PS)

PSPath = Path.from_compound(PS)

replace2connection = CrosslinkerGeometry(
    bead_name="Bbone",
    connection_sites=[6, 7],
    coordinates=PSPath.coordinates,
    bond_graph=PSPath.bond_graph
)
G = nx.Graph().add_edges_from([(0,1)])
replace3connection = CrosslinkerGeometry(
    bead_name="Bbone",
    connection_sites=[0,0,1],
    coordinates=np.array([[0,0,0], [0,0,0.3]]),
    bond_graph=G
)
path.replace_sites(replacement=(replace2connection,replace3connection), sites=np.arange(10))
path.visualize(0.2)


ImportError: cannot import name 'replace_sites' from 'mbuild.path.crosslink' (/Users/calcraven/Dropbox/Mac/Documents/Vanderbilt/Research/MoSDeF/test_mosdef_repos/mbuild-2.0/polyethylene-casestudy2/pkgs/mbuild/mbuild/path/crosslink.py)

In [13]:
# 9. Demonstrate excluded bond_depth

path = build_parallel_linear_paths(5, [1,2,3])

cl = CrosslinkerGeometry.single_site("_CROSS", n_connections=4)

crosslink(
    path, crosslinker=cl, 
    backbone_name=("A","A", "C","C"), 
    crosslink_bond_length=1, tolerance=0.2,
    excluded_bond_depth=0, minimum_separation=0.1
    )
    
crosslink( # works
    path, crosslinker=cl, 
    backbone_name=("E","E", "G","G"), 
    crosslink_bond_length=2, tolerance=0.1,
    excluded_bond_depth=2, minimum_separation=0.1
    )

path.visualize(radius=0.4)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# 10. Crosslink to two neighboring backbone sites

path = build_parallel_linear_paths(5)

cl = CrosslinkerGeometry.single_site("_CROSS", n_connections=2)

crosslink(
    path, crosslinker=cl, 
    backbone_name=(("A","B"), ("C","D")), 
    crosslink_bond_length=1, tolerance=0.2,
    excluded_bond_depth=0,
    )
    
path.visualize(radius=0.4)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.